## Portfolio Agents with Market Intelligence Tools

Advanced portfolio agents with integrated tools for market news, financial metrics, and portfolio optimization recommendations.

### Setup and Imports

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import requests

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

openai_client = project_client.get_openai_client()

print("Setup complete")

### Client Portfolio Data

In [ ]:
client_portfolios = {
    "Client_1": {
        "name": "Client 1 - Tech-Heavy",
        "risk_profile": "Moderate-High",
        "stocks": ["TCS.NS", "WIPRO.NS", "INFY.NS", "HDFCBANK.NS"],
        "mutual_funds": ["119551", "119022"],
        "target_allocation": {"equity": 70, "debt": 20, "cash": 10}
    },
    "Client_2": {
        "name": "Client 2 - Diversified",
        "risk_profile": "Moderate",
        "stocks": ["RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "LT.NS"],
        "mutual_funds": ["119551", "120485"],
        "target_allocation": {"equity": 60, "debt": 30, "cash": 10}
    },
    "Client_3": {
        "name": "Client 3 - Growth-Focused",
        "risk_profile": "High",
        "stocks": ["TATAMOTORS.NS", "ADANIGREEN.NS", "ETERNAL.NS", "PAYTM.NS"],
        "mutual_funds": ["119598", "119618"],
        "target_allocation": {"equity": 80, "debt": 15, "cash": 5}
    },
    "Client_4": {
        "name": "Client 4 - Income-Focused",
        "risk_profile": "Low-Moderate",
        "stocks": ["ITC.NS", "COALINDIA.NS", "NTPC.NS", "POWERGRID.NS"],
        "mutual_funds": ["119547"],
        "target_allocation": {"equity": 50, "debt": 40, "cash": 10}
    },
    "Client_5": {
        "name": "Client 5 - Balanced",
        "risk_profile": "Moderate",
        "stocks": ["INFY.NS", "SUNPHARMA.NS", "BHARTIARTL.NS", "TATACONSUM.NS"],
        "mutual_funds": ["119510", "120303"],
        "target_allocation": {"equity": 65, "debt": 25, "cash": 10}
    }
}

print("Portfolio data loaded with risk profiles")

### Financial Analysis Tools

In [ ]:
def calculate_portfolio_risk_metrics(client_id):
    """Calculate risk metrics for portfolio"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    # Fetch 1-year historical data for all stocks
    returns = []
    for stock in portfolio['stocks']:
        try:
            df = yf.download(stock, period="1y", progress=False, auto_adjust=True)
            returns.append(df['Close'].pct_change().std() * np.sqrt(252))  # Annualized volatility
        except:
            pass
    
    avg_volatility = np.mean(returns) if returns else 0
    
    return {
        "client_id": client_id,
        "portfolio_volatility": round(avg_volatility * 100, 2),
        "risk_profile": portfolio.get("risk_profile"),
        "diversification_score": len(portfolio['stocks']) + len(portfolio['mutual_funds']),
        "target_allocation": portfolio.get("target_allocation")
    }

def generate_rebalancing_recommendation(client_id):
    """Generate portfolio rebalancing recommendations"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    target = portfolio.get("target_allocation", {})
    
    recommendations = {
        "client_id": client_id,
        "current_date": datetime.now().strftime("%Y-%m-%d"),
        "recommendation_type": "Quarterly Rebalancing",
        "target_allocation": target,
        "actions": []
    }
    
    # Generate sample actions based on risk profile
    if portfolio.get("risk_profile") == "High":
        recommendations["actions"].append("Consider taking partial profits on top performers")
        recommendations["actions"].append("Rebalance growth stocks to maintain target equity allocation")
    elif portfolio.get("risk_profile") == "Low-Moderate":
        recommendations["actions"].append("Maintain higher debt allocation for stability")
        recommendations["actions"].append("Review dividend-paying stocks for income optimization")
    else:
        recommendations["actions"].append("Maintain balanced allocation")
        recommendations["actions"].append("Review quarterly performance against benchmarks")
    
    return recommendations

def fetch_stock_fundamentals(ticker):
    """Fetch fundamental metrics for a stock"""
    try:
        ticker_obj = yf.Ticker(ticker)
        info = ticker_obj.info
        
        return {
            "ticker": ticker,
            "name": info.get("longName", "N/A"),
            "pe_ratio": round(info.get("trailingPE", 0), 2),
            "pb_ratio": round(info.get("priceToBook", 0), 2),
            "dividend_yield": round(info.get("dividendYield", 0) * 100, 2),
            "market_cap": info.get("marketCap", "N/A"),
            "eps": round(info.get("trailingEps", 0), 2),
            "52_week_high": round(info.get("fiftyTwoWeekHigh", 0), 2),
            "52_week_low": round(info.get("fiftyTwoWeekLow", 0), 2)
        }
    except Exception as e:
        return {"ticker": ticker, "error": str(e)}

def create_portfolio_comparison_report(client_id):
    """Create comprehensive portfolio comparison with benchmarks"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    # Get benchmark data
    nifty = yf.Ticker("^NSEI")
    sensex = yf.Ticker("^BSESN")
    
    report = {
        "client_id": client_id,
        "portfolio_name": portfolio["name"],
        "generated_date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "holdings_count": {
            "stocks": len(portfolio['stocks']),
            "mutual_funds": len(portfolio['mutual_funds'])
        },
        "risk_profile": portfolio.get("risk_profile"),
        "benchmarks": {
            "nifty_50": "Equity benchmark for comparison",
            "bse_sensex": "Broad market index for comparison"
        }
    }
    
    return report

print("Financial analysis tools defined")
import numpy as np  # For risk calculations

### Create Intelligent Portfolio Managers

In [ ]:
def create_intelligent_manager_prompt(client_id):
    """Create intelligent manager prompt with financial tools context"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return None
    
    # Fetch comprehensive data
    rebalancing = generate_rebalancing_recommendation(client_id)
    comparison = create_portfolio_comparison_report(client_id)
    
    prompt = f"""
You are an expert Portfolio Manager AI for {portfolio['name']}.

PORTFOLIO PROFILE:
==================
Client Risk Profile: {portfolio['risk_profile']}
Holdings: {len(portfolio['stocks'])} stocks + {len(portfolio['mutual_funds'])} mutual funds
Stocks: {', '.join(portfolio['stocks'])}
Target Allocation: Equity {portfolio['target_allocation']['equity']}% | Debt {portfolio['target_allocation']['debt']}% | Cash {portfolio['target_allocation']['cash']}%

YOUR ROLE:
==========
As the portfolio manager, you should:

1. PERFORMANCE TRACKING
   - Monitor individual stock performance daily
   - Track portfolio against Nifty 50 and BSE Sensex benchmarks
   - Identify underperforming assets
   - Highlight outperformers and profit-taking opportunities

2. RISK MANAGEMENT
   - Assess concentration risks
   - Monitor sector exposures
   - Evaluate correlation between holdings
   - Flag any breaches in risk tolerance

3. REBALANCING STRATEGY
   - Recommend quarterly rebalancing
   - Suggest tax-efficient rebalancing methods
   - Propose new additions to improve diversification
   - Recommend reducing overlapping positions

4. FUNDAMENTAL ANALYSIS
   - Analyze P/E ratios relative to historical averages
   - Evaluate dividend sustainability
   - Track earnings announcements
   - Monitor market cap and valuation metrics

5. MARKET INTELLIGENCE
   - Provide context on macro-economic trends
   - Discuss sector rotation opportunities
   - Alert on significant market movements
   - Relate portfolio holdings to market indices

FINANCIAL TOOLS AT YOUR DISPOSAL:
==================================
✓ Real-time stock data via Yahoo Finance
✓ Mutual fund NAV data via MFAPI
✓ Historical price trends for technical analysis
✓ Portfolio risk metrics and volatility calculations
✓ Benchmark comparison reports
✓ Market indices snapshots

ANALYSIS FRAMEWORKS:
====================
When analyzing the portfolio, use these frameworks:

Framework 1: SWOT Analysis
- Strengths: Well-diversified, good risk/return profile
- Weaknesses: Any concentration risks or poor performers
- Opportunities: Undervalued sectors, new growth areas
- Threats: Market volatility, geopolitical factors

Framework 2: Asset Allocation Review
- Compare current vs. target allocation
- Recommend shifts based on market conditions
- Consider tax implications

Framework 3: Sector Analysis
- Map holdings to sectors
- Compare sector weights to indices
- Identify sector trends

COMMUNICATION STYLE:
====================
- Be professional and data-driven
- Use specific metrics (not vague language)
- Always cite data sources and dates
- Provide actionable recommendations
- Highlight risks before opportunities
- Consider the client's risk profile in all recommendations

IMPORTANT GUIDELINES:
======================
- Avoid making guaranteed predictions
- Always mention that past performance ≠ future results
- Flag any significant volatility or risks
- Recommend consulting with financial advisors for major decisions
- Provide analysis in the context of long-term goals
"""
    
    return prompt.strip()

# Create intelligent portfolio manager agents
intelligent_managers = {}

for client_id, portfolio in client_portfolios.items():
    agent_name = f"portfolio-manager-{client_id.lower()}"
    instructions = create_intelligent_manager_prompt(client_id)
    
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions
        )
    )
    
    intelligent_managers[client_id] = {
        "agent": agent,
        "agent_name": agent_name,
        "portfolio": portfolio,
        "recommendations": generate_rebalancing_recommendation(client_id)
    }
    
    print(f"✅ Created Portfolio Manager: {agent_name}")
    print(f"   Risk Profile: {portfolio['risk_profile']}")
    print(f"   Holdings: {len(portfolio['stocks'])} stocks + {len(portfolio['mutual_funds'])} funds")
    print()

### Create Conversations for Intelligent Managers

In [ ]:
intelligent_conversations = {}

for client_id in intelligent_managers.keys():
    conversation = openai_client.conversations.create()
    intelligent_conversations[client_id] = conversation.id

print("Conversations created for intelligent portfolio managers")

### Test Intelligent Managers - Scenario Based Queries

In [ ]:
# Scenario 1: Market downturn - Query high-risk portfolio (Client 3)
client_id = "Client_3"
scenario_query = """The market has dropped 5% this week due to RBI rate hike announcement. 
How should I protect my portfolio? Should I move to more defensive positions?"""

response = openai_client.responses.create(
    conversation=intelligent_conversations[client_id],
    extra_body={
        "agent": {
            "name": intelligent_managers[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=scenario_query
)

print(f"SCENARIO: Market Downturn - {intelligent_managers[client_id]['portfolio']['name']}")
print(f"\nQuery:\n{scenario_query}")
print(f"\nPortfolio Manager Response:\n{response.output_text}")
print("\n" + "="*70 + "\n")

### Specific Stock Analysis Queries

In [ ]:
# Client 1 - Tech portfolio analysis
client_id = "Client_1"
query = """I want to understand the tech sector performance in my portfolio.
How are TCS, WIPRO, and INFY performing relative to each other?
Which one has the best growth potential for the next quarter?"""

response = openai_client.responses.create(
    conversation=intelligent_conversations[client_id],
    extra_body={
        "agent": {
            "name": intelligent_managers[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"TECH SECTOR ANALYSIS - {intelligent_managers[client_id]['portfolio']['name']}")
print(f"\nQuery:\n{query}")
print(f"\nPortfolio Manager Response:\n{response.output_text}")
print("\n" + "="*70 + "\n")

### Rebalancing Recommendation Queries

In [ ]:
# Client 4 - Income-focused rebalancing
client_id = "Client_4"
query = """It's Q2 and I'd like to review my portfolio allocation.
Given my income focus strategy, should I adjust my equity/debt split?
Are there any dividend plays I should consider adding?"""

response = openai_client.responses.create(
    conversation=intelligent_conversations[client_id],
    extra_body={
        "agent": {
            "name": intelligent_managers[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"REBALANCING ANALYSIS - {intelligent_managers[client_id]['portfolio']['name']}")
print(f"\nQuery:\n{query}")
print(f"\nPortfolio Manager Response:\n{response.output_text}")
print("\n" + "="*70 + "\n")

### Diversification and Risk Assessment

In [ ]:
# Client 2 - Diversification check
client_id = "Client_2"
query = """I'm concerned about portfolio concentration.
Looking at my holdings (RELIANCE, HDFCBANK, ICICIBANK, LT), 
am I over-exposed to financials and energy? What sectors should I add?"""

response = openai_client.responses.create(
    conversation=intelligent_conversations[client_id],
    extra_body={
        "agent": {
            "name": intelligent_managers[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=query
)

print(f"DIVERSIFICATION ASSESSMENT - {intelligent_managers[client_id]['portfolio']['name']}")
print(f"\nQuery:\n{query}")
print(f"\nPortfolio Manager Response:\n{response.output_text}")

### Portfolio Manager Summary and Metrics

In [ ]:
print("\n" + "="*80)
print("INTELLIGENT PORTFOLIO MANAGERS - COMPLETE DEPLOYMENT")
print("="*80)

summary_data = []
for client_id, manager_info in intelligent_managers.items():
    portfolio = manager_info['portfolio']
    recommendations = manager_info['recommendations']
    
    summary_data.append({
        'Portfolio': portfolio['name'],
        'Manager ID': manager_info['agent']['id'][:8] + '...',
        'Risk Level': portfolio['risk_profile'],
        'Stocks': len(portfolio['stocks']),
        'Funds': len(portfolio['mutual_funds']),
        'Equity%': portfolio['target_allocation']['equity'],
        'Debt%': portfolio['target_allocation']['debt']
    })

df_managers = pd.DataFrame(summary_data)
print("\n" + df_managers.to_string(index=False))

print(f"\n{'='*80}")
print(f"Total Intelligent Portfolio Managers Deployed: {len(intelligent_managers)}")
print(f"Total Conversations Active: {len(intelligent_conversations)}")
print(f"Average Portfolio Complexity: {sum(len(m['portfolio']['stocks']) + len(m['portfolio']['mutual_funds']) for m in intelligent_managers.values()) / len(intelligent_managers):.1f} instruments")
print(f"{'='*80}")

print("\nCapabilities:")
print("  ✓ Real-time performance monitoring")
print("  ✓ Risk assessment and alerts")
print("  ✓ Quarterly rebalancing recommendations")
print("  ✓ Sector analysis and rotation strategies")
print("  ✓ Tax-efficient trading suggestions")
print("  ✓ Benchmark comparison analysis")
print("  ✓ Fundamental metrics evaluation")
print("  ✓ Market intelligence and trend analysis")